# Отчёт по лабораторной работе №1 на тему «Введение в Spring Boot»

**Выполнил:** студент группы 12002408

Блинов Кирилл Игоревич

**Проверил:** ассистент кафедры информационных и робототехнических систем

Пузырев Никита Сергеевич

**Дисциплина:** Разработка кроссплатформенных приложений, семестр 2

[![Открыть в Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kir1lka/cad-2025/blob/main/sem2/lab1/rep.ipynb)

## Цель работы

Получить практические навыки работы со Spring Boot, создать веб-приложение
для хранения заметок и проверить обработку HTTP-запросов.

## Задание для выполнения

1. Изучить теоретический материал.
2. Создать приложение по примеру из методички.
3. Реализовать добавление, просмотр, изменение и удаление сообщений.
4. Выполнить два дополнительных задания: очистку всех сообщений и подсчёт их количества, пункты 1 и 6.
5. Подготовить отчёт.

Задание: [Работа 1 Spring.pdf](https://github.com/kir1lka/cad-2025/blob/main/sem2/lab1/Работа%201%20Spring.pdf).

## Ход работы

### 1. Создание проекта

Создан отдельный Gradle-проект `messages`. Использованы Java 17, Gradle 8.12
и Spring Boot 3.2.3. Версия Spring Boot соответствует примеру из методички.
В предыдущих лабораторных также использовался Gradle. Методичка допускает
использование Maven или Gradle.

Основные файлы проекта:

```text
lab1/
  build.gradle
  settings.gradle
  gradlew
  gradlew.bat
  gradle/wrapper/
  src/main/java/ru/bsuedu/cad/lab/
    MessagesApplication.java
    MessageController.java
  src/main/resources/application.properties
  src/test/java/ru/bsuedu/cad/lab/MessageControllerTest.java
```

Ниже приведены выполняемые ячейки Python: они собирают и запускают Java-приложение,
после чего отправляют HTTP-запросы. Код приложения написан на Java.
Сохранённые результаты получены при локальном выполнении этих ячеек,
запуск в облачной среде Colab отдельно не проверялся.
Для повторения в Colab нужно выполнить ячейки по порядку.

In [1]:
import atexit
import json
import os
from pathlib import Path
import shutil
import socket
import subprocess
import tempfile
import time
import urllib.error
import urllib.request
import xml.etree.ElementTree as ET

try:
    import google.colab
    in_colab = True
except ImportError:
    in_colab = False

if in_colab:
    checkout = Path('/content/cad-2025-lab1')
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/kir1lka/cad-2025.git', str(checkout)], check=True)
    project = checkout / 'sem2' / 'lab1'
    java_home = Path('/usr/lib/jvm/java-17-openjdk-amd64')
    if not (java_home / 'bin/java').exists():
        subprocess.run(['apt-get', 'update', '-qq'], check=True)
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-17-jdk-headless'], check=True)
    os.environ['JAVA_HOME'] = str(java_home)
    os.environ['PATH'] = str(java_home / 'bin') + os.pathsep + os.environ['PATH']
else:
    project = Path(os.environ.get('LAB1_PROJECT_DIR', '.')).resolve()
    if not (project / 'build.gradle').exists():
        project = project / 'sem2' / 'lab1'

assert (project / 'build.gradle').is_file(), 'Укажите папку проекта в LAB1_PROJECT_DIR'
gradle = [str(project / 'gradlew.bat')] if os.name == 'nt' else ['bash', str(project / 'gradlew')]
java = str(Path(os.environ['JAVA_HOME']) / 'bin' / ('java.exe' if os.name == 'nt' else 'java')) if os.environ.get('JAVA_HOME') else shutil.which('java')
assert java, 'Для запуска нужен JDK 17'
print('Проект messages найден')
print('Сборка: Gradle Wrapper 8.12')

Проект messages найден
Сборка: Gradle Wrapper 8.12


### 2. Настройка Gradle

`spring-boot-starter-web` подключает Spring MVC и встроенный Tomcat.
`spring-boot-starter-test` используется для проверки обработчиков запросов.
Плагин Spring Boot собирает исполняемый JAR, а dependency-management
подбирает согласованные версии зависимостей.

```groovy
plugins {
    id 'java'
    id 'org.springframework.boot' version '3.2.3'
    id 'io.spring.dependency-management' version '1.1.4'
}

group = 'ru.bsuedu.cad.lab'
version = '1.0'

java {
    sourceCompatibility = JavaVersion.VERSION_17
    targetCompatibility = JavaVersion.VERSION_17
}

repositories {
    mavenCentral()
}

dependencies {
    implementation 'org.springframework.boot:spring-boot-starter-web'
    testImplementation 'org.springframework.boot:spring-boot-starter-test'
}

tasks.named('test') {
    useJUnitPlatform()
}

tasks.withType(JavaCompile).configureEach {
    options.encoding = 'UTF-8'
    options.compilerArgs.add('-parameters')
}
```

### 3. Точка входа

Аннотация `@SpringBootApplication` включает конфигурацию, автоматическую настройку
и поиск компонентов. Метод `SpringApplication.run` запускает приложение.

```java
package ru.bsuedu.cad.lab;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class MessagesApplication {

    public static void main(String[] args) {
        SpringApplication.run(MessagesApplication.class, args);
    }
}
```

### 4. Контроллер сообщений

`@RestController` обозначает класс, который обрабатывает запросы и возвращает
данные в теле ответа. Сообщения хранятся в `ArrayList` в памяти приложения.
`@RequestBody` получает текст сообщения, а `@PathVariable` получает индекс из адреса.
При перезапуске приложения список очищается. Индексы начинаются с нуля
и сдвигаются после удаления элемента.

Методы синхронизированы, чтобы параллельные запросы не изменяли список одновременно.
При чтении возвращается копия списка.

```java
package ru.bsuedu.cad.lab;

import org.springframework.web.bind.annotation.DeleteMapping;
import org.springframework.web.bind.annotation.GetMapping;
import org.springframework.web.bind.annotation.PathVariable;
import org.springframework.web.bind.annotation.PostMapping;
import org.springframework.web.bind.annotation.PutMapping;
import org.springframework.web.bind.annotation.RequestBody;
import org.springframework.web.bind.annotation.RestController;

import java.util.ArrayList;
import java.util.List;

@RestController
public class MessageController {

    private final List<String> userMessages = new ArrayList<>();

    @GetMapping("/")
    public String helloWorld() {
        return "Hello, World!";
    }

    @GetMapping("/messages")
    public synchronized List<String> getAllMessages() {
        return new ArrayList<>(userMessages);
    }

    @PostMapping("/messages")
    public synchronized String publishMessage(@RequestBody String message) {
        userMessages.add(message);
        return "Message published successfully!";
    }

    @PutMapping("/messages/{index}")
    public synchronized String updateMessage(@PathVariable int index,
                                             @RequestBody String message) {
        if (index >= 0 && index < userMessages.size()) {
            userMessages.set(index, message);
            return "Message updated successfully!";
        }
        return "Message not found at index " + index;
    }

    @DeleteMapping("/messages/{index}")
    public synchronized String deleteMessage(@PathVariable int index) {
        if (index >= 0 && index < userMessages.size()) {
            userMessages.remove(index);
            return "Message deleted successfully!";
        }
        return "Message not found at index " + index;
    }

    @DeleteMapping("/messages")
    public synchronized String clearAllMessages() {
        userMessages.clear();
        return "All messages deleted successfully!";
    }

    @GetMapping("/messages/count")
    public synchronized int countMessages() {
        return userMessages.size();
    }
}
```

### 5. Сборка и автоматическая проверка

Тесты выполняют запросы через MockMvc и проверяют приветствие, пустой список,
цикл добавления, изменения и удаления, очистку и подсчёт сообщений.
Отдельно проверяются отрицательный индекс, нулевой индекс пустого списка
и индекс за пределами списка. Всего выполняется 10 тестовых случаев.

In [2]:
result = subprocess.run(gradle + ['--no-daemon', '--console=plain', 'test', 'bootJar'],
                        cwd=project, capture_output=True, text=True, encoding='utf-8', errors='replace')
if result.returncode:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError('Ошибка сборки')
for line in result.stdout.splitlines():
    if line.startswith('> Task') or line.startswith('BUILD SUCCESSFUL'):
        print(line)
suites = [ET.parse(path).getroot() for path in (project / 'build/test-results/test').glob('TEST-*.xml')]
assert suites, 'Результаты тестов не найдены'
tests = sum(int(suite.attrib['tests']) for suite in suites)
failures = sum(int(suite.attrib['failures']) for suite in suites)
errors = sum(int(suite.attrib['errors']) for suite in suites)
skipped = sum(int(suite.attrib['skipped']) for suite in suites)
print(f'Тестов: {tests}, ошибок: {errors}, неудачных: {failures}, пропущено: {skipped}')
assert tests == 10 and failures == errors == skipped == 0

> Task :compileJava
> Task :processResources
> Task :classes
> Task :resolveMainClassName
> Task :bootJar
> Task :compileTestJava
> Task :processTestResources NO-SOURCE
> Task :testClasses
> Task :test
BUILD SUCCESSFUL in 12s
Тестов: 10, ошибок: 0, неудачных: 0, пропущено: 0


### 6. Запуск приложения

Для обычного запуска из папки проекта в PowerShell используется команда:

```powershell
.\gradlew.bat bootRun
```

В этом режиме приложение доступно по адресу `http://localhost:8080`.
Ячейка ниже запускает собранный JAR на свободном локальном порту,
чтобы проверка не мешала другим запущенным приложениям.

In [3]:
with socket.socket() as probe:
    probe.bind(('127.0.0.1', 0))
    port = probe.getsockname()[1]
base_url = f'http://127.0.0.1:{port}'
jars = [path for path in (project / 'build/libs').glob('*.jar') if not path.name.endswith('-plain.jar')]
assert len(jars) == 1, 'Исполняемый JAR не найден или найдено несколько JAR'
server_log = tempfile.TemporaryFile(mode='w+', encoding='utf-8')
server = subprocess.Popen([java, '-Dfile.encoding=UTF-8', '-jar', str(jars[0]),
                           f'--server.port={port}', '--server.address=127.0.0.1'],
                          cwd=project, stdout=server_log, stderr=subprocess.STDOUT)

def stop_server():
    if server.poll() is None:
        server.terminate()
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait(timeout=5)

atexit.register(stop_server)
for attempt in range(180):
    if server.poll() is not None:
        break
    try:
        with urllib.request.urlopen(base_url + '/', timeout=1) as response:
            assert response.read().decode('utf-8') == 'Hello, World!'
        print('Приложение Spring Boot запущено')
        break
    except (OSError, urllib.error.URLError):
        time.sleep(0.25)
else:
    stop_server()
    raise RuntimeError('Приложение не запустилось за отведённое время')
if server.poll() is not None:
    server_log.seek(0)
    raise RuntimeError(server_log.read())

def request(method, path, body=None):
    data = body.encode('utf-8') if body is not None else None
    req = urllib.request.Request(base_url + path, data=data, method=method,
                                 headers={'Content-Type': 'text/plain; charset=utf-8'})
    with urllib.request.urlopen(req, timeout=10) as response:
        status = response.status
        text = response.read().decode('utf-8')
    print(f'{method} {path} -> HTTP {status}: {text}')
    assert status == 200
    return text

Приложение Spring Boot запущено


### 7. Основные операции

Проверено приветствие и получение пустого списка. Затем добавлено сообщение
`Первая заметка`, проверено его наличие, выполнено изменение на
`Изменённая заметка` и удаление по индексу 0.

In [4]:
assert request('GET', '/') == 'Hello, World!'
assert json.loads(request('GET', '/messages')) == []
assert request('POST', '/messages', 'Первая заметка') == 'Message published successfully!'
assert json.loads(request('GET', '/messages')) == ['Первая заметка']
assert request('PUT', '/messages/0', 'Изменённая заметка') == 'Message updated successfully!'
assert json.loads(request('GET', '/messages')) == ['Изменённая заметка']
assert request('DELETE', '/messages/0') == 'Message deleted successfully!'
assert json.loads(request('GET', '/messages')) == []

GET / -> HTTP 200: Hello, World!
GET /messages -> HTTP 200: []
POST /messages -> HTTP 200: Message published successfully!
GET /messages -> HTTP 200: ["Первая заметка"]
PUT /messages/0 -> HTTP 200: Message updated successfully!
GET /messages -> HTTP 200: ["Изменённая заметка"]
DELETE /messages/0 -> HTTP 200: Message deleted successfully!
GET /messages -> HTTP 200: []


### 8. Дополнительные задания

Реализованы два дополнительных метода:

1. `clearAllMessages()` очищает список по запросу `DELETE /messages`.
2. `countMessages()` возвращает количество сообщений по запросу `GET /messages/count`.

Для проверки добавлены две заметки. До очистки количество равно 2,
после очистки список пустой и количество равно 0. Повторная очистка также допустима.

In [5]:
request('POST', '/messages', 'Заметка один')
request('POST', '/messages', 'Заметка два')
assert json.loads(request('GET', '/messages')) == ['Заметка один', 'Заметка два']
assert int(request('GET', '/messages/count')) == 2
assert request('DELETE', '/messages') == 'All messages deleted successfully!'
assert json.loads(request('GET', '/messages')) == []
assert int(request('GET', '/messages/count')) == 0
request('DELETE', '/messages')

POST /messages -> HTTP 200: Message published successfully!
POST /messages -> HTTP 200: Message published successfully!
GET /messages -> HTTP 200: ["Заметка один","Заметка два"]
GET /messages/count -> HTTP 200: 2
DELETE /messages -> HTTP 200: All messages deleted successfully!
GET /messages -> HTTP 200: []
GET /messages/count -> HTTP 200: 0
DELETE /messages -> HTTP 200: All messages deleted successfully!


### 9. Проверка неверных индексов

При попытке изменить или удалить отсутствующее сообщение возвращается пояснение,
а список остаётся пустым. Ответ с кодом 200 и текстом ошибки сохранён по примеру
методички. Проверены отрицательный индекс и индекс за пределами списка.

In [6]:
assert request('PUT', '/messages/-1', 'Нет сообщения') == 'Message not found at index -1'
assert request('DELETE', '/messages/10') == 'Message not found at index 10'
assert json.loads(request('GET', '/messages')) == []

PUT /messages/-1 -> HTTP 200: Message not found at index -1
DELETE /messages/10 -> HTTP 200: Message not found at index 10
GET /messages -> HTTP 200: []


### 10. Остановка приложения

После проверки процесс Java завершается. Для остановки приложения,
запущенного через `bootRun` в терминале, используется сочетание Ctrl+C.

In [7]:
stop_server()
server_log.close()
atexit.unregister(stop_server)
print('Приложение остановлено')

Приложение остановлено


## Вывод

В ходе лабораторной работы создано приложение Spring Boot для хранения заметок.
Реализованы основные операции с сообщениями, очистка списка и подсчёт его элементов.
Проект собран через Gradle. Все 10 автоматических тестов прошли успешно.
Проверены HTTP-запросы к запущенному приложению и обработка неверных индексов.
Отчёт сохранён в формате ноутбука, который можно открыть в Google Colab.

## Использованные материалы

1. Методичка «Работа 1 Spring.pdf» из репозитория преподавателя.
2. [Spring Boot 3.2.3: Getting Started](https://docs.spring.io/spring-boot/docs/3.2.3/reference/html/getting-started.html).
3. [Пример оформления отчёта в dap-2024](https://github.com/kir1lka/dap-2024/blob/main/les02/rep.ipynb).